<a href="https://colab.research.google.com/github/Adarshtrajan/CVAI/blob/Big-Data/Adarsh_%7C_Big_Data_Analytics_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Installing dependencies
!pip install pyspark
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

# Setting up Spark environment
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

In [ ]:
# Importing necessary libraries
import pandas as pd
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg

# Uploading file
from google.colab import files
uploaded = files.upload()

# Providing the path for reading the dataset
dataset_path = "/content/store_customers.csv"

# Load dataset into Pandas for initial verification
df = pd.read_csv(dataset_path)
print("First few rows of the dataset:")
print(df.head())

Saving store_customers.csv to store_customers (1).csv
First few rows of the dataset:
   CustomerID  Age  Salary  Gender  Country
0           1   72   20000    Male  Germany
1           2   72   22000  Female   France
2           3   70   24000  Female  England
3           4   75    2600    Male  England
4           5   33   50000    Male   France


Hadoop MapReduce Job

In [ ]:
# Detecting anomalies in Salary

def map_function(data):
    """Maps input records to key-value pairs."""
    for row in data:
        customer_id, age, salary, gender, country = row
        salary = int(salary)
        if salary < 5000:  # Define an anomaly
            yield (customer_id, f"Anomaly detected: Salary = {salary}")

def reduce_function(mapped_data):
    """Aggregates mapped data."""
    anomalies = {}
    for key, value in mapped_data:
        anomalies[key] = value
    return anomalies

# Running MapReduce job (simulation)
mapped_data = map_function(df.values)
anomalies_detected = reduce_function(mapped_data)
print("\nAnomalies detected in Salary:")
for k, v in anomalies_detected.items():
    print(f"Customer {k}: {v}")


Anomalies detected in Salary:
Customer 4: Anomaly detected: Salary = 2600
Customer 7: Anomaly detected: Salary = 4300


 Apache Spark Job

In [ ]:
# Start Spark session
spark = SparkSession.builder.appName("StoreCustomersAnalysis").getOrCreate()

# Load dataset into Spark DataFrame
spark_df = spark.read.csv(dataset_path, header=True, inferSchema=True)

# Show schema
spark_df.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Salary: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Country: string (nullable = true)



In [ ]:
# Aggregating average salary by country
avg_salary_by_country = spark_df.groupBy("Country").agg(avg("Salary").alias("Avg_Salary"))
print("\nAverage Salary by Country:")
avg_salary_by_country.show()


Average Salary by Country:
+-------+-----------------+
|Country|       Avg_Salary|
+-------+-----------------+
|Germany|35345.32258064516|
| France|35394.37404967055|
|England|35364.16166719293|
+-------+-----------------+



In [ ]:
# Filtering customers above 70 years of age
older_customers = spark_df.filter(col("Age") > 70)
print("\nCustomers above 70 years:")
older_customers.show()


Customers above 70 years:
+----------+---+------+------+-------+
|CustomerID|Age|Salary|Gender|Country|
+----------+---+------+------+-------+
|         1| 72| 20000|  Male|Germany|
|         2| 72| 22000|Female| France|
|         4| 75|  2600|  Male|England|
|         9| 76| 35000|  Male|Germany|
|        32| 76| 37000|  Male|England|
|        39| 71|  7600|  Male|England|
|        44| 73| 40000|Female|England|
|        45| 75| 42000|  Male|England|
|        46| 73| 30000|Female|England|
|        52| 75| 14300|Female|England|
|        56| 74| 35000|Female|England|
|        68| 73| 37000|Female|England|
|        80| 80| 47000|Female|England|
|        99| 78| 12600|  Male|England|
|       108| 74| 39000|Female|England|
|       118| 73| 37000|  Male|England|
|       136| 80| 50000|  Male| France|
|       152| 75| 14300|Female|Germany|
|       160| 73| 65000|  Male|England|
|       164| 78|  7600|  Male|England|
+----------+---+------+------+-------+
only showing top 20 rows



In [ ]:
# Basic EDA
print("\nSummary Statistics:")
spark_df.describe().show()


Summary Statistics:
+-------+------------------+-----------------+------------------+------+-------+
|summary|        CustomerID|              Age|            Salary|Gender|Country|
+-------+------------------+-----------------+------------------+------+-------+
|  count|              7000|             7000|              7000|  7000|   7000|
|   mean|            3500.5|           50.261|35367.671428571426|  NULL|   NULL|
| stddev|2020.8702745764426|17.31340482725415|15002.106029438502|  NULL|   NULL|
|    min|                 1|               20|              2600|Female|England|
|    max|              7000|               80|             65000|  Male|Germany|
+-------+------------------+-----------------+------------------+------+-------+



In [ ]:
# Stopping the Spark session
spark.stop()

In [ ]:
import time
from pyspark.sql import SparkSession

# Initialize Spark session
spark = SparkSession.builder.appName("PerformanceComparison").getOrCreate()

# Path to the dataset in Google Colab
dataset_path = "/content/store_customers.csv"

# Load the dataset into Spark DataFrame
df = spark.read.csv(dataset_path, header=True, inferSchema=True)

### Measuring Hadoop MapReduce Execution Time ###
start_time_mapreduce = time.time()

# Simulating MapReduce job for anomaly detection (low salary detection)
def map_function(row):
    if row['Salary'] < 5000:  # Identifying anomalies
        return [(row['CustomerID'], row['Salary'])]
    return []

# Applying map function
mapreduce_results = df.rdd.flatMap(map_function).collect()

end_time_mapreduce = time.time()
mapreduce_execution_time = end_time_mapreduce - start_time_mapreduce

### Measuring Spark Execution Time ###
start_time_spark = time.time()

# Performing salary aggregation (Average Salary by Country)
spark_results = df.groupBy("Country").agg({"Salary": "avg"}).collect()

end_time_spark = time.time()
spark_execution_time = end_time_spark - start_time_spark

### Comparing Performance ###
print("\n=== Performance Comparison ===")
print(f"Hadoop MapReduce Execution Time: {mapreduce_execution_time:.4f} seconds")
print(f"Apache Spark Execution Time: {spark_execution_time:.4f} seconds")

# Scalability and Ease of Use Analysis
scalability_comparison = """
MapReduce: Requires complex job setup, manual handling of parallelism, and is slower due to disk I/O operations.
Spark: Faster due to in-memory processing, optimized for iterative computations, and provides easier APIs for data analysis.
"""
print(scalability_comparison)


=== Performance Comparison ===
Hadoop MapReduce Execution Time: 1.8219 seconds
Apache Spark Execution Time: 0.6174 seconds

MapReduce: Requires complex job setup, manual handling of parallelism, and is slower due to disk I/O operations.
Spark: Faster due to in-memory processing, optimized for iterative computations, and provides easier APIs for data analysis.



Advanced Analytics and Machine Learning

In [ ]:
# Importing the necessary libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Initializing the Spark session
spark = SparkSession.builder.appName("MLlib_Classification").getOrCreate()

# Loading the dataset
dataset_path = "/content/store_customers.csv"
df = spark.read.csv(dataset_path, header=True, inferSchema=True)

# Displaying dataset schema
df.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Salary: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Country: string (nullable = true)



In [ ]:
# Selecting relevant features for prediction (Age, Salary → Predict Gender)
df = df.select("Age", "Salary", "Gender")

# Encoding the Gender as an integer (Male → 1, Female → 0)
indexer = StringIndexer(inputCol="Gender", outputCol="Gender_Index")
df = indexer.fit(df).transform(df)

# Assembling the features into a single column
feature_assembler = VectorAssembler(inputCols=["Age", "Salary"], outputCol="features")
df = feature_assembler.transform(df).select("features", "Gender_Index")

# Splitting the data into Train (80%) and Test (20%) sets
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

# Training the Logistic Regression Model
lr = LogisticRegression(labelCol="Gender_Index", featuresCol="features")
model = lr.fit(train_data)

# Making the predictions
predictions = model.transform(test_data)

# Evaluating the model using Accuracy metric
evaluator = MulticlassClassificationEvaluator(labelCol="Gender_Index", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)

In [ ]:
# Displaying the results
print("\n=== Model Performance ===")
print(f"Accuracy: {accuracy:.4f}")

# Showing Sample Predictions
predictions.select("features", "Gender_Index", "prediction").show(10)


=== Model Performance ===
Accuracy: 0.5243
+--------------+------------+----------+
|      features|Gender_Index|prediction|
+--------------+------------+----------+
| [20.0,9300.0]|         0.0|       0.0|
|[20.0,12600.0]|         0.0|       0.0|
|[20.0,14300.0]|         0.0|       0.0|
|[20.0,19300.0]|         0.0|       0.0|
|[20.0,29000.0]|         0.0|       0.0|
|[20.0,30000.0]|         0.0|       0.0|
|[20.0,32000.0]|         0.0|       0.0|
|[20.0,34000.0]|         1.0|       0.0|
|[20.0,37000.0]|         1.0|       0.0|
|[20.0,37000.0]|         1.0|       0.0|
+--------------+------------+----------+
only showing top 10 rows

